<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


---

<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://sebastianraschka.com">Sebastian Raschka</a> 所著《<a href="http://mng.bz/orYv">从零构建大语言模型</a>》一书的补充代码<br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px" alt="书籍封面"></a>
</td>
</tr>
</table>


# Byte Pair Encoding (BPE) Tokenizer From Scratch -- Simple


---

# 从零开始构建字节对编码（BPE）分词器 -- 简单版


- This is a standalone notebook implementing the popular byte pair encoding (BPE) tokenization algorithm, which is used in models like GPT-2 to GPT-4, Llama 3, etc., from scratch for educational purposes
- For more details about the purpose of tokenization, please refer to [Chapter 2](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb); this code here is bonus material explaining the BPE algorithm
- The original BPE tokenizer that OpenAI implemented for training the original GPT models can be found [here](https://github.com/openai/gpt-2/blob/master/src/encoder.py)
- The BPE algorithm was originally described in 1994: "[A New Algorithm for Data Compression](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf)" by Philip Gage
- Most projects, including Llama 3, nowadays use OpenAI's open-source [tiktoken library](https://github.com/openai/tiktoken) due to its computational performance; it allows loading pretrained GPT-2 and GPT-4 tokenizers, for example (the Llama 3 models were trained using the GPT-4 tokenizer as well)
- The difference between the implementations above and my implementation in this notebook, besides it being is that it also includes a function for training the tokenizer (for educational purposes)
- There's also an implementation called [minBPE](https://github.com/karpathy/minbpe) with training support, which is maybe more performant (my implementation here is focused on educational purposes); in contrast to `minbpe` my implementation additionally allows loading the original OpenAI tokenizer vocabulary and merges


---

- 这是一个独立的笔记本，从零开始实现了流行的字节对编码（BPE）分词算法，该算法被用于GPT-2到GPT-4、Llama 3等模型中，旨在用于教育目的
- 关于分词目的的更多细节，请参阅[第2章](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb)；此处的代码是解释BPE算法的补充材料
- OpenAI为训练原始GPT模型实现的原始BPE分词器可以在[这里](https://github.com/openai/gpt-2/blob/master/src/encoder.py)找到
- BPE算法最初于1994年由Philip Gage在《[一种新的数据压缩算法](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf)》中描述
- 目前大多数项目（包括Llama 3）由于计算性能原因使用OpenAI的开源[tiktoken库](https://github.com/openai/tiktoken)；例如，它允许加载预训练的GPT-2和GPT-4分词器（Llama 3模型也是使用GPT-4分词器训练的）
- 上述实现与本笔记本中我的实现之间的区别，除了它之外，还包括一个用于训练分词器的函数（用于教育目的）
- 还有一个名为[minBPE](https://github.com/karpathy/minbpe)的实现支持训练，可能性能更好（我这里的实现侧重于教育目的）；与`minbpe`相比，我的实现额外允许加载原始的OpenAI分词器词汇表和合并规则


**This is a very naive implementation for educational purposes. The [bpe-from-scratch.ipynb](bpe-from-scratch.ipynb) notebook contains a more sophisticated (but much harder to read) implementation that matches the behavior in tiktoken.**


---

**这是一个用于教学目的的简单实现。[bpe-from-scratch.ipynb](bpe-from-scratch.ipynb) 笔记本包含了一个更复杂（但也更难理解）的实现，其行为与tiktoken中的行为一致。**


&nbsp;
# 1. The main idea behind byte pair encoding (BPE)


---

&nbsp;
# 1. 字节对编码（BPE）的核心思想


- The main idea in BPE is to convert text into an integer representation (token IDs) for LLM training (see [Chapter 2](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb))

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/bpe-overview.webp" width="600px">


---

- BPE的核心思想是将文本转换为整数表示（token ID）以用于大语言模型训练（参见[第2章](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb)）

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/bpe-overview.webp" width="600px">


&nbsp;
## 1.1 Bits and bytes


---

&nbsp;
## 1.1 比特与字节


- Before getting to the BPE algorithm, let's introduce the notion of bytes
- Consider converting text into a byte array (BPE stands for "byte" pair encoding after all):


---

- 在介绍BPE算法之前，我们先引入字节的概念
- 考虑将文本转换为字节数组（毕竟BPE代表"字节对编码"）：


In [1]:
text = "This is some text"
byte_ary = bytearray(text, "utf-8")
print(byte_ary)

bytearray(b'This is some text')


- When we call `list()` on a `bytearray` object, each byte is treated as an individual element, and the result is a list of integers corresponding to the byte values:


---

- 当我们对 `bytearray` 对象调用 `list()` 时，每个字节被视为独立元素，结果是一个与字节值对应的整数列表：


In [2]:
ids = list(byte_ary)
print(ids)

[84, 104, 105, 115, 32, 105, 115, 32, 115, 111, 109, 101, 32, 116, 101, 120, 116]


- This would be a valid way to convert text into a token ID representation that we need for the embedding layer of an LLM
- However, the downside of this approach is that it is creating one ID for each character (that's a lot of IDs for a short text!)
- I.e., this means for a 17-character input text, we have to use 17 token IDs as input to the LLM:


---

- 这是将文本转换为LLM嵌入层所需的token ID表示的有效方法
- 但这种方法的缺点是为每个字符创建一个ID（对于短文本来说ID数量太多！）
- 也就是说，对于17个字符的输入文本，我们需要使用17个token ID作为LLM的输入：


In [3]:
print("Number of characters:", len(text))
print("Number of token IDs:", len(ids))

Number of characters: 17
Number of token IDs: 17


- If you have worked with LLMs before, you may know that the BPE tokenizers have a vocabulary where we have a token ID for whole words or subwords instead of each character
- For example, the GPT-2 tokenizer tokenizes the same text ("This is some text") into only 4 instead of 17 tokens: `1212, 318, 617, 2420`
- You can double-check this using the interactive [tiktoken app](https://tiktokenizer.vercel.app/?model=gpt2) or the [tiktoken library](https://github.com/openai/tiktoken):

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/tiktokenizer.webp" width="600px">

```python
import tiktoken

gpt2_tokenizer = tiktoken.get_encoding("gpt2")
gpt2_tokenizer.encode("This is some text")
# prints [1212, 318, 617, 2420]
```


---

- 如果您之前使用过大语言模型，可能知道 BPE 分词器的词汇表中，每个词元 ID 对应的是完整单词或子词，而非单个字符
- 例如，GPT-2 分词器将相同文本（"This is some text"）仅分为 4 个词元而非 17 个：`1212, 318, 617, 2420`
- 您可以通过交互式 [tiktoken 应用](https://tiktokenizer.vercel.app/?model=gpt2) 或 [tiktoken 库](https://github.com/openai/tiktoken) 进行验证：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/tiktokenizer.webp" width="600px">

```python
import tiktoken

gpt2_tokenizer = tiktoken.get_encoding("gpt2")
gpt2_tokenizer.encode("This is some text")
# 输出 [1212, 318, 617, 2420]
```


- Since a byte consists of 8 bits, there are 2<sup>8</sup> = 256 possible values that a single byte can represent, ranging from 0 to 255
- You can confirm this by executing the code `bytearray(range(0, 257))`, which will warn you that `ValueError: byte must be in range(0, 256)`)
- A BPE tokenizer usually uses these 256 values as its first 256 single-character tokens; one could visually check this by running the following code:

```python
import tiktoken
gpt2_tokenizer = tiktoken.get_encoding("gpt2")

for i in range(300):
    decoded = gpt2_tokenizer.decode([i])
    print(f"{i}: {decoded}")
"""
prints:
0: !
1: "
2: #
...
255: �  # <---- single character tokens up to here
256:  t
257:  a
...
298: ent
299:  n
"""
```


---

- 由于一个字节由8位组成，因此单个字节可表示的可能值有2<sup>8</sup> = 256个，范围从0到255
- 您可以通过执行代码 `bytearray(range(0, 257))` 来验证这一点，该代码会警告 `ValueError: byte must be in range(0, 256)`
- BPE分词器通常将这256个值作为其前256个单字符词元；可以通过运行以下代码直观地验证：

```python
import tiktoken
gpt2_tokenizer = tiktoken.get_encoding("gpt2")

for i in range(300):
    decoded = gpt2_tokenizer.decode([i])
    print(f"{i}: {decoded}")
"""
输出：
0: !
1: "
2: #
...
255: �  # <---- 单字符词元到此为止
256:  t
257:  a
...
298: ent
299:  n
"""
```


- Above, note that entries 256 and 257 are not single-character values but double-character values (a whitespace + a letter), which is a little shortcoming of the original GPT-2 BPE Tokenizer (this has been improved in the GPT-4 tokenizer)


---

- 注意，上述第256和257个条目并非单字符值，而是双字符值（一个空格字符加一个字母），这是原始GPT-2 BPE分词器的一个小缺陷（此问题已在GPT-4分词器中得到改进）。


&nbsp;
## 1.2 Building the vocabulary


---

&nbsp;
## 1.2 构建词汇表


- The goal of the BPE tokenization algorithm is to build a vocabulary of commonly occurring subwords like `298: ent` (which can be found in *entangle, entertain, enter, entrance, entity, ...*, for example), or even complete words like 

```
318: is
617: some
1212: This
2420: text
```


---

- BPE分词算法的目标是构建一个包含常见子词的词汇表，例如 `298: ent`（可在 *entangle, entertain, enter, entrance, entity, ...* 等单词中找到），甚至包括完整单词如

```
318: is
617: some
1212: This
2420: text
```


- The BPE algorithm was originally described in 1994: "[A New Algorithm for Data Compression](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf)" by Philip Gage
- Before we get to the actual code implementation, the form that is used for LLM tokenizers today can be summarized as follows:


---

- BPE算法最初于1994年由Philip Gage在论文《[一种新的数据压缩算法](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf)》中提出
- 在进入实际代码实现之前，当今大语言模型分词器所采用的形式可概括如下：


&nbsp;
## 1.3 BPE algorithm outline

**1. Identify frequent pairs**
- In each iteration, scan the text to find the most commonly occurring pair of bytes (or characters)

**2. Replace and record**

- Replace that pair with a new placeholder ID (one not already in use, e.g., if we start with 0...255, the first placeholder would be 256)
- Record this mapping in a lookup table
- The size of the lookup table is a hyperparameter, also called "vocabulary size" (for GPT-2, that's
50,257)

**3. Repeat until no gains**

- Keep repeating steps 1 and 2, continually merging the most frequent pairs
- Stop when no further compression is possible (e.g., no pair occurs more than once)

**Decompression (decoding)**

- To restore the original text, reverse the process by substituting each ID with its corresponding pair, using the lookup table


---

&nbsp;
## 1.3 BPE 算法概述

**1. 识别高频对**
- 在每次迭代中，扫描文本以找到最常出现的字节对（或字符对）

**2. 替换并记录**
- 用新的占位符 ID 替换该对（使用尚未使用的 ID，例如，如果我们从 0...255 开始，第一个占位符将是 256）
- 在查找表中记录此映射
- 查找表的大小是一个超参数，也称为“词汇表大小”（对于 GPT-2，该值为 50,257）

**3. 重复直至无收益**
- 持续重复步骤 1 和 2，不断合并出现频率最高的对
- 当无法进一步压缩时停止（例如，没有一对出现超过一次）

**解压缩（解码）**
- 要恢复原始文本，请使用查找表将每个 ID 替换为其对应的对，从而逆转该过程


&nbsp;
## 1.4 BPE algorithm example

### 1.4.1 Concrete example of the encoding part (steps 1 & 2)

- Suppose we have the text (training dataset) `the cat in the hat` from which we want to build the vocabulary for a BPE tokenizer

**Iteration 1**

1. Identify frequent pairs
  - In this text, "th" appears twice (at the beginning and before the second "e")

2. Replace and record
  - replace "th" with a new token ID that is not already in use, e.g., 256
  - the new text is: `<256>e cat in <256>e hat`
  - the new vocabulary is

```
  0: ...
  ...
  256: "th"
```

**Iteration 2**

1. **Identify frequent pairs**  
   - In the text `<256>e cat in <256>e hat`, the pair `<256>e` appears twice

2. **Replace and record**  
   - replace `<256>e` with a new token ID that is not already in use, for example, `257`.  
   - The new text is:
     ```
     <257> cat in <257> hat
     ```
   - The updated vocabulary is:
     ```
     0: ...
     ...
     256: "th"
     257: "<256>e"
     ```

**Iteration 3**

1. **Identify frequent pairs**  
   - In the text `<257> cat in <257> hat`, the pair `<257> ` appears twice (once at the beginning and once before “hat”).

2. **Replace and record**  
   - replace `<257> ` with a new token ID that is not already in use, for example, `258`.  
   - the new text is:
     ```
     <258>cat in <258>hat
     ```
   - The updated vocabulary is:
     ```
     0: ...
     ...
     256: "th"
     257: "<256>e"
     258: "<257> "
     ```
     
- and so forth

&nbsp;
### 1.4.2 Concrete example of the decoding part (steps 3)

- To restore the original text, we reverse the process by substituting each token ID with its corresponding pair in the reverse order they were introduced
- Start with the final compressed text: `<258>cat in <258>hat`
-  Substitute `<258>` → `<257> `: `<257> cat in <257> hat`  
- Substitute `<257>` → `<256>e`: `<256>e cat in <256>e hat`
- Substitute `<256>` → "th": `the cat in the hat`


---

&nbsp;
## 1.4 BPE算法示例

### 1.4.1 编码部分的具体示例（步骤1和2）

- 假设我们有文本（训练数据集）`the cat in the hat`，我们想用它为BPE分词器构建词汇表

**迭代1**

1. 识别高频对
   - 在此文本中，"th"出现两次（在开头和第二个"e"之前）

2. 替换并记录
   - 将"th"替换为一个尚未使用的新token ID，例如256
   - 新文本为：`<256>e cat in <256>e hat`
   - 新词汇表为

```
  0: ...
  ...
  256: "th"
```

**迭代2**

1. **识别高频对**  
   - 在文本`<256>e cat in <256>e hat`中，对`<256>e`出现两次

2. **替换并记录**  
   - 将`<256>e`替换为一个尚未使用的新token ID，例如`257`。  
   - 新文本为：
     ```
     <257> cat in <257> hat
     ```
   - 更新后的词汇表为：
     ```
     0: ...
     ...
     256: "th"
     257: "<256>e"
     ```

**迭代3**

1. **识别高频对**  
   - 在文本`<257> cat in <257> hat`中，对`<257> `出现两次（一次在开头，一次在"hat"之前）。

2. **替换并记录**  
   - 将`<257> `替换为一个尚未使用的新token ID，例如`258`。  
   - 新文本为：
     ```
     <258>cat in <258>hat
     ```
   - 更新后的词汇表为：
     ```
     0: ...
     ...
     256: "th"
     257: "<256>e"
     258: "<257> "
     ```
     
- 依此类推

&nbsp;
### 1.4.2 解码部分的具体示例（步骤3）

- 要恢复原始文本，我们通过反向替换过程，按照引入的逆序将每个token ID替换为其对应的对
- 从最终压缩文本开始：`<258>cat in <258>hat`
- 将`<258>`替换为`<257> `：`<257> cat in <257> hat`  
- 将`<257>`替换为`<256>e`：`<256>e cat in <256>e hat`
- 将`<256>`替换为"th"：`the cat in the hat`


&nbsp;
## 2. A simple BPE implementation


---

&nbsp;
## 2. 一个简单的BPE实现


- Below is an implementation of this algorithm described above as a Python class that mimics the `tiktoken` Python user interface
- Note that the encoding part above describes the original training step via `train()`; however, the `encode()` method works similarly (although it looks a bit more complicated because of the special token handling):

1. Split the input text into individual bytes
2. Repeatedly find & replace (merge) adjacent tokens (pairs) when they match any pair in the learned BPE merges (from highest to lowest "rank," i.e., in the order they were learned)
3. Continue merging until no more merges can be applied
4. The final list of token IDs is the encoded output


---

- 以下是上述算法的实现，以Python类的形式呈现，模拟了`tiktoken`的Python用户界面
- 注意：上面的编码部分描述了通过`train()`进行的原始训练步骤；然而，`encode()`方法的工作方式类似（尽管由于特殊标记的处理，看起来稍微复杂一些）：

1. 将输入文本拆分为单个字节
2. 当相邻标记（对）与学习到的BPE合并中的任何对匹配时，反复查找并替换（合并）它们（按从高到低的"等级"顺序，即按学习顺序进行）
3. 持续合并，直到无法再应用任何合并
4. 最终的标记ID列表即为编码输出


In [ ]:
from collections import Counter, deque
from functools import lru_cache


class BPETokenizerSimple:
    def __init__(self):
        # Maps token_id to token_str (e.g., {11246: "some"})
        self.vocab = {}
        # Maps token_str to token_id (e.g., {"some": 11246})
        self.inverse_vocab = {}
        # Dictionary of BPE merges: {(token_id1, token_id2): merged_token_id}
        self.bpe_merges = {}

    def train(self, text, vocab_size, allowed_special={"<|endoftext|>"}):
        """
        Train the BPE tokenizer from scratch.

        Args:
            text (str): The training text.
            vocab_size (int): The desired vocabulary size.
            allowed_special (set): A set of special tokens to include.
        """

        # Preprocess: Replace spaces with 'Ġ'
        # Note that Ġ is a particularity of the GPT-2 BPE implementation
        # E.g., "Hello world" might be tokenized as ["Hello", "Ġworld"]
        # (GPT-4 BPE would tokenize it as ["Hello", " world"])
        processed_text = []
        for i, char in enumerate(text):
            if char == " " and i != 0:
                processed_text.append("Ġ")
            if char != " ":
                processed_text.append(char)
        processed_text = "".join(processed_text)

        # Initialize vocab with unique characters, including 'Ġ' if present
        # Start with the first 256 ASCII characters
        unique_chars = [chr(i) for i in range(256)]

        # Extend unique_chars with characters from processed_text that are not already included
        unique_chars.extend(char for char in sorted(set(processed_text)) if char not in unique_chars)

        # Optionally, ensure 'Ġ' is included if it is relevant to your text processing
        if 'Ġ' not in unique_chars:
            unique_chars.append('Ġ')

        # Now create the vocab and inverse vocab dictionaries
        self.vocab = {i: char for i, char in enumerate(unique_chars)}
        self.inverse_vocab = {char: i for i, char in self.vocab.items()}

        # Add allowed special tokens
        if allowed_special:
            for token in allowed_special:
                if token not in self.inverse_vocab:
                    new_id = len(self.vocab)
                    self.vocab[new_id] = token
                    self.inverse_vocab[token] = new_id

        # Tokenize the processed_text into token IDs
        token_ids = [self.inverse_vocab[char] for char in processed_text]

        # BPE steps 1-3: Repeatedly find and replace frequent pairs
        for new_id in range(len(self.vocab), vocab_size):
            if len(token_ids) < 2:
                break
            pair_id = self.find_freq_pair(token_ids, mode="most")
            if pair_id is None:  # No more pairs to merge. Stopping training.
                break
            
            updated = self.replace_pair(token_ids, pair_id, new_id)
            if updated == token_ids:
                break

            token_ids = updated
            self.bpe_merges[pair_id] = new_id

        # Build the vocabulary with merged tokens
        for (p0, p1), new_id in self.bpe_merges.items():
            merged_token = self.vocab[p0] + self.vocab[p1]
            self.vocab[new_id] = merged_token
            self.inverse_vocab[merged_token] = new_id

    def encode(self, text):
        """
        Encode the input text into a list of token IDs.

        Args:
            text (str): The text to encode.

        Returns:
            List[int]: The list of token IDs.
        """
        tokens = []
        # Split text into tokens, keeping newlines intact
        words = text.replace("\n", " \n ").split()  # Ensure '\n' is treated as a separate token

        for i, word in enumerate(words):
            if i > 0 and not word.startswith("\n"):
                tokens.append("Ġ" + word)  # Add 'Ġ' to words that follow a space or newline
            else:
                tokens.append(word)  # Handle first word or standalone '\n'

        token_ids = []
        for token in tokens:
            if token in self.inverse_vocab:
                # token is contained in the vocabulary as is
                token_id = self.inverse_vocab[token]
                token_ids.append(token_id)
            else:
                # Attempt to handle subword tokenization via BPE
                sub_token_ids = self.tokenize_with_bpe(token)
                token_ids.extend(sub_token_ids)

        return token_ids

    def tokenize_with_bpe(self, token):
        """
        Tokenize a single token using BPE merges.

        Args:
            token (str): The token to tokenize.

        Returns:
            List[int]: The list of token IDs after applying BPE.
        """
        # Tokenize the token into individual characters (as initial token IDs)
        token_ids = [self.inverse_vocab.get(char, None) for char in token]
        if None in token_ids:
            missing_chars = [char for char, tid in zip(token, token_ids) if tid is None]
            raise ValueError(f"Characters not found in vocab: {missing_chars}")

        can_merge = True
        while can_merge and len(token_ids) > 1:
            can_merge = False
            new_tokens = []
            i = 0
            while i < len(token_ids) - 1:
                pair = (token_ids[i], token_ids[i + 1])
                if pair in self.bpe_merges:
                    merged_token_id = self.bpe_merges[pair]
                    new_tokens.append(merged_token_id)
                    # Uncomment for educational purposes:
                    # print(f"Merged pair {pair} -> {merged_token_id} ('{self.vocab[merged_token_id]}')")
                    i += 2  # Skip the next token as it's merged
                    can_merge = True
                else:
                    new_tokens.append(token_ids[i])
                    i += 1
            if i < len(token_ids):
                new_tokens.append(token_ids[i])
            token_ids = new_tokens

        return token_ids

    def decode(self, token_ids):
        """
        Decode a list of token IDs back into a string.

        Args:
            token_ids (List[int]): The list of token IDs to decode.

        Returns:
            str: The decoded string.
        """
        decoded_string = ""
        for token_id in token_ids:
            if token_id not in self.vocab:
                raise ValueError(f"Token ID {token_id} not found in vocab.")
            token = self.vocab[token_id]
            if token.startswith("Ġ"):
                # Replace 'Ġ' with a space
                decoded_string += " " + token[1:]
            else:
                decoded_string += token
        return decoded_string

    @lru_cache(maxsize=None)
    def get_special_token_id(self, token):
        return self.inverse_vocab.get(token, None)

    @staticmethod
    def find_freq_pair(token_ids, mode="most"):
        if(len(token_ids) < 2):
            return None
        pairs = Counter(zip(token_ids, token_ids[1:]))
        if not pairs:
            return None

        if mode == "most":
            return max(pairs.items(), key=lambda x: x[1])[0]
        elif mode == "least":
            return min(pairs.items(), key=lambda x: x[1])[0]
        else:
            raise ValueError("Invalid mode. Choose 'most' or 'least'.")

    @staticmethod
    def replace_pair(token_ids, pair_id, new_id):
        dq = deque(token_ids)
        replaced = []

        while dq:
            current = dq.popleft()
            if dq and (current, dq[0]) == pair_id:
                replaced.append(new_id)
                # Remove the 2nd token of the pair, 1st was already removed
                dq.popleft()
            else:
                replaced.append(current)

        return replaced


### Edge-case handling for short token sequences

BPE merges require adjacent token pairs.  
If the token sequence has fewer than 2 items, no pair exists, so `find_freq_pair` returns `None` and training stops gracefully.


---

### 短token序列的边缘情况处理

BPE合并需要相邻的token对。  
如果token序列少于2个元素，则不存在配对，因此`find_freq_pair`会返回`None`，训练将正常终止。


In [21]:
tok = BPETokenizerSimple()

assert tok.find_freq_pair([]) is None
assert tok.find_freq_pair([42]) is None

tok.train("", vocab_size=270)
tok.train("H", vocab_size=270)
tok.train("He", vocab_size=270)

print("Edge-case checks passed.")

Edge-case checks passed.


- There is a lot of code in the `BPETokenizerSimple` class above, and discussing it in detail is out of scope for this notebook, but the next section offers a short overview of the usage to understand the class methods a bit better


---

- 上面的 `BPETokenizerSimple` 类中包含大量代码，详细讨论已超出本 notebook 的范围，但下一节将简要概述其用法，以便更好地理解该类的方法。


## 3. BPE implementation walkthrough


---

## 3. BPE 实现详解


- In practice, I highly recommend using [tiktoken](https://github.com/openai/tiktoken) as my implementation above focuses on readability and educational purposes, not on performance
- However, the usage is more or less similar to tiktoken, except that tiktoken does not have a training method
- Let's see how my `BPETokenizerSimple` Python code above works by looking at some examples below (a detailed code discussion is out of scope for this notebook)


---

- 在实践中，我强烈建议使用 [tiktoken](https://github.com/openai/tiktoken)，因为我的上述实现侧重于可读性和教学目的，而非性能优化
- 不过，其用法与 tiktoken 大致相似，区别在于 tiktoken 不包含训练方法
- 通过以下示例，让我们看看上述 `BPETokenizerSimple` Python 代码是如何工作的（详细的代码讨论不在本笔记范围内）


### 3.1 Training, encoding, and decoding


---

### 3.1 训练、编码与解码


- First, let's consider some sample text as our training dataset:


---

首先，让我们将一些示例文本视为我们的训练数据集。


In [5]:
import os
import urllib.request

if not os.path.exists("../01_main-chapter-code/the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    file_path = "../01_main-chapter-code/the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)

with open("../01_main-chapter-code/the-verdict.txt", "r", encoding="utf-8") as f: # added ../01_main-chapter-code/
    text = f.read()

- Next, let's initialize and train the BPE tokenizer with a vocabulary size of 1,000
- Note that the vocabulary size is already 255 by default due to the byte values discussed earlier, so we are only "learning" 745 vocabulary entries 
- For comparison, the GPT-2 vocabulary is 50,257 tokens, the GPT-4 vocabulary is 100,256 tokens (`cl100k_base` in tiktoken), and GPT-4o uses 199,997 tokens (`o200k_base` in tiktoken); they have all much bigger training sets compared to our simple example text above


---

- 接下来，让我们初始化并训练词汇量为1,000的BPE分词器
- 需要注意的是，由于之前讨论的字节值，默认词汇量已为255，因此我们实际上只“学习”了745个词汇条目
- 作为对比，GPT-2的词汇量为50,257个token，GPT-4的词汇量为100,256个token（在tiktoken中为`cl100k_base`），而GPT-4o使用199,997个token（在tiktoken中为`o200k_base`）；与我们上述的简单示例文本相比，它们都拥有大得多的训练数据集


In [6]:
tokenizer = BPETokenizerSimple()
tokenizer.train(text, vocab_size=1000, allowed_special={"<|endoftext|>"})

- You may want to inspect the vocabulary contents (but note it will create a long list)


---

- 您可能想要查看词汇内容（但请注意这会生成一个很长的列表）


In [7]:
# print(tokenizer.vocab)
print(len(tokenizer.vocab))

1000


- This vocabulary is created by merging 742 times (~ `1000 - len(range(0, 256))`)


---

- 该词表通过合并742次生成（约 `1000 - len(range(0, 256))`）


In [8]:
print(len(tokenizer.bpe_merges))

742


- This means that the first 256 entries are single-character tokens


---

- 这意味着前256个条目是单字符标记


- Next, let's use the created merges via the `encode` method to encode some text:


---

- 接下来，让我们通过 `encode` 方法使用已创建的合并规则来编码一些文本：


In [9]:
input_text = "Jack embraced beauty through art and life."
token_ids = tokenizer.encode(input_text)
print(token_ids)

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46]


In [10]:
print("Number of characters:", len(input_text))
print("Number of token IDs:", len(token_ids))

Number of characters: 42
Number of token IDs: 20


- From the lengths above, we can see that a 42-character sentence was encoded into 20 token IDs, effectively cutting the input length roughly in half compared to a character-byte-based encoding


---

- 从上述长度可以看出，一个42个字符的句子被编码为20个token ID，与基于字符字节的编码相比，输入长度大致减半。


- Note that the vocabulary itself is used in the `decode()` method, which allows us to map the token IDs back into text:


---

- 请注意，词汇表本身在 `decode()` 方法中使用，这使我们能够将 token ID 映射回文本：


In [11]:
print(token_ids)

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46]


In [12]:
print(tokenizer.decode(token_ids))

Jack embraced beauty through art and life.


- Iterating over each token ID can give us a better understanding of how the token IDs are decoded via the vocabulary:


---

- 遍历每个 token ID 能让我们更好地理解 token ID 如何通过词汇表进行解码：


In [13]:
for token_id in token_ids:
    print(f"{token_id} -> {tokenizer.decode([token_id])}")

424 -> Jack
256 ->  
654 -> em
531 -> br
302 -> ac
311 -> ed
256 ->  
296 -> be
97 -> a
465 -> ut
121 -> y
595 ->  through
841 ->  ar
116 -> t
287 ->  a
466 -> nd
256 ->  
326 -> li
972 -> fe
46 -> .


- As we can see, most token IDs represent 2-character subwords; that's because the training data text is very short with not that many repetitive words, and because we used a relatively small vocabulary size


---

- 我们可以看到，大多数token ID代表的是2字符的子词；这是因为训练数据文本非常短，重复词汇不多，且我们使用了相对较小的词汇量。


- As a summary, calling `decode(encode())` should be able to reproduce arbitrary input texts:


---

- 总结来说，调用 `decode(encode())` 应能重现任意输入文本：


In [14]:
tokenizer.decode(tokenizer.encode("This is some text."))

'This is some text.'

&nbsp;
# 4. Conclusion


---

&nbsp;
# 4. 结论


- That's it! That's how BPE works in a nutshell, complete with a training method for creating new tokenizers 
- I hope you found this brief tutorial useful for educational purposes; if you have any questions, please feel free to open a new Discussion [here](https://github.com/rasbt/LLMs-from-scratch/discussions/categories/q-a)


**This is a very naive implementation for educational purposes. The [bpe-from-scratch.ipynb](bpe-from-scratch.ipynb) notebook contains a more sophisticated (but much harder to read) implementation that matches the behavior in tiktoken.**


---

- 以上就是BPE（字节对编码）的工作原理简述，包括创建新分词器的训练方法
- 希望这篇简短教程对您的学习有所帮助；若有任何疑问，欢迎在[此处](https://github.com/rasbt/LLMs-from-scratch/discussions/categories/q-a)发起讨论

**这是一个用于教学目的的非常基础的实现。[bpe-from-scratch.ipynb](bpe-from-scratch.ipynb)笔记本中包含更复杂的实现（但可读性较差），其行为与tiktoken保持一致。**
